In [0]:
from pyspark.sql import functions as F

SILVER_TABLE = "credit_card_project.silver.cleaned_transactions"
TARGET_TABLE = "credit_card_project.gold.gold_card_behavior"


BRONZE_CARD_TABLE = "credit_card_project.bronze.cards"

silver = spark.table(SILVER_TABLE)
bronze_cards = spark.table(BRONZE_CARD_TABLE)

print("Silver table:", SILVER_TABLE)
print("Bronze card table:", BRONZE_CARD_TABLE)

print("\nSilver schema:")
silver.printSchema()

print("\nBronze card schema:")
bronze_cards.printSchema()

print("\nSilver row count:", silver.count())
print("Bronze card row count:", bronze_cards.count())


### Aggregate Silver transactions by card

In [0]:
card_transaction_agg = (
    silver #silver transaction table
    .groupBy("user_id", "card_id")
    .agg(
        F.count("*")
            .cast("long")
            .alias("total_transactions"),

        F.sum("amount")
            .cast("decimal(18,2)")
            .alias("total_spend"),

        F.avg("amount")
            .cast("decimal(18,4)")
            .alias("avg_transaction_amount"),

        F.countDistinct("merchant_id")
            .cast("long")
            .alias("unique_merchants"),

        F.countDistinct("merchant_category_code")
            .cast("long")
            .alias("unique_mccs"),

        F.sum(
            F.when(F.col("is_fraud") == True, 1).otherwise(0)
        )
        .cast("long")
        .alias("fraud_count"),

        F.sum(
            F.when(F.col("is_online") == True, 1).otherwise(0)
        )
        .cast("long")
        .alias("_online_count"),

        F.sum(
            F.when(
                F.dayofweek("transaction_timestamp").isin(1, 7),
                1
            ).otherwise(0)
        )
        .cast("long")
        .alias("_weekend_count"),

        F.sum(
            F.when(
                F.hour("transaction_timestamp").between(0, 5),
                1
            ).otherwise(0)
        )
        .cast("long")
        .alias("_night_count")
    )
    .withColumn(
        "fraud_rate",
        (
            F.col("fraud_count") /
            F.col("total_transactions")
        ).cast("double")
    )
    .withColumn(
        "online_transaction_pct",
        (
            F.col("_online_count") * F.lit(100.0) /
            F.col("total_transactions")
        ).cast("double")
    )
    .withColumn(
        "weekend_transaction_pct",
        (
            F.col("_weekend_count") * F.lit(100.0) /
            F.col("total_transactions")
        ).cast("double")
    )
    .withColumn(
        "night_transaction_pct",
        (
            F.col("_night_count") * F.lit(100.0) /
            F.col("total_transactions")
        ).cast("double")
    )
    .drop(
        "_online_count",
        "_weekend_count",
        "_night_count"
    )
)

print("Aggregated card transaction rows:", card_transaction_agg.count())

display(card_transaction_agg.limit(10))


### Build gold_card_behavior

In [0]:
gold_card_behavior = (
    bronze_cards.alias("b")
    .join(
        card_transaction_agg.alias("t"),
        on=[
            F.col("b.user_id").cast("long") == F.col("t.user_id"),
            F.col("b.card_id").cast("long") == F.col("t.card_id")
        ],
        how="left"
    )
    .select(
        F.col("b.user_id").cast("long").alias("user_id"),
        F.col("b.card_id").cast("long").alias("card_id"),
        F.col("b.card_brand").alias("card_brand"),
        F.col("b.card_type").alias("card_type"),
        F.col("b.credit_limit").alias("credit_limit"),
        F.col("b.acct_open_date").alias("acct_open_date"),
        F.col("b.expires").alias("expires"),

        F.coalesce(
            F.col("t.total_transactions"),
            F.lit(0)
        ).cast("long").alias("total_transactions"),

        F.coalesce(
            F.col("t.total_spend"),
            F.lit(0).cast("decimal(18,2)")
        ).cast("decimal(18,2)").alias("total_spend"),

        F.col("t.avg_transaction_amount")
            .cast("decimal(18,4)")
            .alias("avg_transaction_amount"),

        F.coalesce(
            F.col("t.unique_merchants"),
            F.lit(0)
        ).cast("long").alias("unique_merchants"),

        F.coalesce(
            F.col("t.unique_mccs"),
            F.lit(0)
        ).cast("long").alias("unique_mccs"),

        F.coalesce(
            F.col("t.fraud_count"),
            F.lit(0)
        ).cast("long").alias("fraud_count"),

        F.col("t.fraud_rate").cast("double").alias("fraud_rate"),

        F.col("t.online_transaction_pct")
            .cast("double")
            .alias("online_transaction_pct"),

        F.col("t.weekend_transaction_pct")
            .cast("double")
            .alias("weekend_transaction_pct"),

        F.col("t.night_transaction_pct")
            .cast("double")
            .alias("night_transaction_pct")
    )
)

gold_card_behavior.printSchema()

display(gold_card_behavior.limit(20))


LEFT JOIN produced exactly those seven cards with the expected NULL/zero behavior:

In [0]:
inactive_cards = (
    gold_card_behavior
    .filter(F.col("total_transactions") == 0)
    .orderBy("user_id", "card_id")
)

print("Inactive cards:", inactive_cards.count())

display(inactive_cards)


In [0]:
display(
    inactive_cards.select(
        "user_id",
        "card_id",
        "card_brand",
        "card_type",
        "credit_limit",
        "acct_open_date",
        "expires",
        "total_transactions",
        "total_spend",
        "avg_transaction_amount",
        "unique_merchants",
        "unique_mccs",
        "fraud_count",
        "fraud_rate",
        "online_transaction_pct",
        "weekend_transaction_pct",
        "night_transaction_pct"
    )
)


### Validate the complete card population and grain

In [0]:
bronze_card_count = bronze_cards.count()

gold_card_count = gold_card_behavior.count()

gold_distinct_cards = (
    gold_card_behavior
    .select("user_id", "card_id")
    .distinct()
    .count()
)

print("Bronze cards:       ", bronze_card_count)
print("Gold rows:           ", gold_card_count)
print("Gold distinct cards: ", gold_distinct_cards)


In [0]:
bronze_card_keys = (
    bronze_cards
    .select(
        F.col("user_id").cast("long").alias("user_id"),
        F.col("card_id").cast("long").alias("card_id")
    )
    .distinct()
)

gold_card_keys = (
    gold_card_behavior
    .select("user_id", "card_id")
    .distinct()
)

missing_cards = (
    bronze_card_keys
    .join(
        gold_card_keys,
        ["user_id", "card_id"],
        "left_anti"
    )
    .count()
)

unexpected_cards = (
    gold_card_keys
    .join(
        bronze_card_keys,
        ["user_id", "card_id"],
        "left_anti"
    )
    .count()
)

print("Missing Bronze cards:   ", missing_cards)
print("Unexpected Gold cards:  ", unexpected_cards)


### Transaction, spend, and fraud reconciliation

In [0]:
silver_transaction_count = silver.count()

gold_transaction_count = (
    gold_card_behavior
    .agg(F.sum("total_transactions").alias("total_transactions"))
    .first()["total_transactions"]
)

silver_total_spend = (
    silver
    .agg(F.sum("amount").alias("total_spend"))
    .first()["total_spend"]
)

gold_total_spend = (
    gold_card_behavior
    .agg(F.sum("total_spend").alias("total_spend"))
    .first()["total_spend"]
)

silver_fraud_count = (
    silver
    .filter(F.col("is_fraud") == True)
    .count()
)

gold_fraud_count = (
    gold_card_behavior
    .agg(F.sum("fraud_count").alias("fraud_count"))
    .first()["fraud_count"]
)

print("Silver transactions:", silver_transaction_count)
print("Gold transactions:  ", gold_transaction_count)
print("Transaction match:  ", silver_transaction_count == gold_transaction_count)

print()

print("Silver total spend:", silver_total_spend)
print("Gold total spend:  ", gold_total_spend)
print("Spend match:       ", silver_total_spend == gold_total_spend)

print()

print("Silver fraud count:", silver_fraud_count)
print("Gold fraud count:  ", gold_fraud_count)
print("Fraud match:       ", silver_fraud_count == gold_fraud_count)


### Validate percentage ranges and NULL behavior

In [0]:
display(
    gold_card_behavior.select(
        F.min("fraud_rate").alias("min_fraud_rate"),
        F.max("fraud_rate").alias("max_fraud_rate"),

        F.min("online_transaction_pct").alias("min_online_pct"),
        F.max("online_transaction_pct").alias("max_online_pct"),

        F.min("weekend_transaction_pct").alias("min_weekend_pct"),
        F.max("weekend_transaction_pct").alias("max_weekend_pct"),

        F.min("night_transaction_pct").alias("min_night_pct"),
        F.max("night_transaction_pct").alias("max_night_pct")
    )
)


In [0]:
display(
    gold_card_behavior.select(
        F.sum(
            F.when(F.col("fraud_rate").isNull(), 1).otherwise(0)
        ).alias("null_fraud_rate"),

        F.sum(
            F.when(F.col("online_transaction_pct").isNull(), 1).otherwise(0)
        ).alias("null_online_pct"),

        F.sum(
            F.when(F.col("weekend_transaction_pct").isNull(), 1).otherwise(0)
        ).alias("null_weekend_pct"),

        F.sum(
            F.when(F.col("night_transaction_pct").isNull(), 1).otherwise(0)
        ).alias("null_night_pct"),

        F.sum(
            F.when(F.col("avg_transaction_amount").isNull(), 1).otherwise(0)
        ).alias("null_avg_amount")
    )
)


### Write gold_card_behavior

In [0]:
(
    gold_card_behavior
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"SUCCESS: wrote {TARGET_TABLE}")


In [0]:
gold_card = spark.table(TARGET_TABLE)

print("Target table:", TARGET_TABLE)

gold_card.printSchema()

print("Gold row count:", gold_card.count())


The change from non-nullable to nullable on persisted columns is normal Delta/Spark behavior and doesn't indicate a data problem.

### Final persisted-table validation

In [0]:
from pyspark.sql import functions as F

gold = spark.table(TARGET_TABLE)

# ------------------------------------------------------------
# 1. Grain and row count
# ------------------------------------------------------------

gold_row_count = gold.count()

gold_distinct_card_count = (
    gold
    .select("user_id", "card_id")
    .distinct()
    .count()
)

duplicate_card_keys = (
    gold
    .groupBy("user_id", "card_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Gold row count:             ", gold_row_count)
print("Distinct card keys:         ", gold_distinct_card_count)
print("Duplicate card keys:        ", duplicate_card_keys)

# ------------------------------------------------------------
# 2. Bronze population reconciliation
# ------------------------------------------------------------

bronze_card_keys = (
    bronze_cards
    .select(
        F.col("user_id").cast("long").alias("user_id"),
        F.col("card_id").cast("long").alias("card_id")
    )
    .distinct()
)

gold_card_keys = (
    gold
    .select("user_id", "card_id")
    .distinct()
)

missing_cards = (
    bronze_card_keys
    .join(gold_card_keys, ["user_id", "card_id"], "left_anti")
    .count()
)

unexpected_cards = (
    gold_card_keys
    .join(bronze_card_keys, ["user_id", "card_id"], "left_anti")
    .count()
)

print("Missing Bronze cards:       ", missing_cards)
print("Unexpected Gold cards:      ", unexpected_cards)

# ------------------------------------------------------------
# 3. Transaction reconciliation
# ------------------------------------------------------------

silver_transaction_count = silver.count()

gold_transaction_count = (
    gold
    .agg(F.sum("total_transactions"))
    .first()[0]
)

print("Silver transactions:        ", silver_transaction_count)
print("Gold transactions:          ", gold_transaction_count)
print("Transaction match:          ",
      silver_transaction_count == gold_transaction_count)

# ------------------------------------------------------------
# 4. Spend reconciliation
# ------------------------------------------------------------

silver_total_spend = (
    silver
    .agg(F.sum("amount"))
    .first()[0]
)

gold_total_spend = (
    gold
    .agg(F.sum("total_spend"))
    .first()[0]
)

print("Silver total spend:         ", silver_total_spend)
print("Gold total spend:           ", gold_total_spend)
print("Spend match:                ",
      silver_total_spend == gold_total_spend)

# ------------------------------------------------------------
# 5. Fraud reconciliation
# ------------------------------------------------------------

silver_fraud_count = (
    silver
    .filter(F.col("is_fraud") == True)
    .count()
)

gold_fraud_count = (
    gold
    .agg(F.sum("fraud_count"))
    .first()[0]
)

print("Silver fraud count:         ", silver_fraud_count)
print("Gold fraud count:           ", gold_fraud_count)
print("Fraud match:                ",
      silver_fraud_count == gold_fraud_count)

# ------------------------------------------------------------
# 6. Inactive-card validation
# ------------------------------------------------------------

inactive_cards = gold.filter(
    F.col("total_transactions") == 0
)

inactive_count = inactive_cards.count()

inactive_bad_rows = (
    inactive_cards
    .filter(
        (F.col("total_spend") != 0) |
        (F.col("unique_merchants") != 0) |
        (F.col("unique_mccs") != 0) |
        (F.col("fraud_count") != 0) |
        F.col("avg_transaction_amount").isNotNull() |
        F.col("fraud_rate").isNotNull() |
        F.col("online_transaction_pct").isNotNull() |
        F.col("weekend_transaction_pct").isNotNull() |
        F.col("night_transaction_pct").isNotNull()
    )
    .count()
)

print("Inactive cards:             ", inactive_count)
print("Invalid inactive rows:      ", inactive_bad_rows)

# ------------------------------------------------------------
# 7. Percentage range validation
# ------------------------------------------------------------

range_check = gold.select(
    F.min("fraud_rate").alias("min_fraud_rate"),
    F.max("fraud_rate").alias("max_fraud_rate"),
    F.min("online_transaction_pct").alias("min_online_pct"),
    F.max("online_transaction_pct").alias("max_online_pct"),
    F.min("weekend_transaction_pct").alias("min_weekend_pct"),
    F.max("weekend_transaction_pct").alias("max_weekend_pct"),
    F.min("night_transaction_pct").alias("min_night_pct"),
    F.max("night_transaction_pct").alias("max_night_pct")
).first()

print("Percentage ranges:")
print(range_check)

# ------------------------------------------------------------
# 8. Final assertions
# ------------------------------------------------------------

assert gold_row_count == 6146
assert gold_distinct_card_count == 6146
assert duplicate_card_keys == 0

assert missing_cards == 0
assert unexpected_cards == 0

assert gold_transaction_count == 24386900
assert gold_transaction_count == silver_transaction_count

assert gold_total_spend == silver_total_spend

assert gold_fraud_count == silver_fraud_count

assert inactive_count == 7
assert inactive_bad_rows == 0

assert range_check["min_fraud_rate"] >= 0
assert range_check["max_fraud_rate"] <= 1

assert range_check["min_online_pct"] >= 0
assert range_check["max_online_pct"] <= 100

assert range_check["min_weekend_pct"] >= 0
assert range_check["max_weekend_pct"] <= 100

assert range_check["min_night_pct"] >= 0
assert range_check["max_night_pct"] <= 100

print()
print("SUCCESS: gold_card_behavior passed all v1 validation checks.")
